✅ 2. Graph Workflow Implementation
Write code or pseudo-code to:

Define state
Add nodes (agents)
Define edges
Implement conditional logic
👉 Must include:

Retry loop if validation fails
Clear start and end states
✅ 3. State Management
Show how state evolves across steps:

Query
Context
Intermediate reasoning
Final answer
Validation flag
✅ 4. Task Delegation & Communication
Demonstrate:

How agents pass information
How decisions are made between agents
🎯 Expected Outcome
A clear multi-step, graph-based agent system that:

Handles complex queries
Demonstrates reasoning + validation
Uses proper orchestration 
Insert your project link :



In [1]:
from typing import TypedDict, Literal
from openai import OpenAI
from langgraph.graph import StateGraph, START, END


client = OpenAI(
    api_key="Api key",
    base_url="https://api.groq.com/openai/v1"
)

def call_llm(system_prompt: str, user_message: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

MAX_RETRIES = 3

print("Testing Groq...")
print(call_llm("You are a helpful assistant.", "Say hello in one sentence."))


class AgentState(TypedDict):
    query                  : str
    category               : str
    context                : str
    intermediate_reasoning : str
    draft_answer           : str
    validation_flag        : bool
    validation_feedback    : str
    retry_count            : int
    final_answer           : str

print("AgentState defined.")


TRANSACTION_DB = {
    "TXN001": {
        "id": "TXN001", "amount": 4500.00, "currency": "INR",
        "status": "completed", "merchant": "Amazon India",
        "date": "2025-03-20", "account": "ACC123",
    },
    "TXN002": {
        "id": "TXN002", "amount": 89000.00, "currency": "INR",
        "status": "flagged", "merchant": "Unknown Merchant",
        "date": "2025-03-22", "account": "ACC456",
        "fraud_flag": True, "flag_reason": "Unusual amount and unknown merchant",
    },
    "TXN003": {
        "id": "TXN003", "amount": 1200.00, "currency": "INR",
        "status": "refund_requested", "merchant": "Flipkart",
        "date": "2025-03-18", "account": "ACC789",
    },
}

FAQ_KB = {
    "upi_limit"    : "UPI limit is Rs 1,00,000 per transaction and Rs 2,00,000 per day.",
    "dispute_time" : "Disputes must be raised within 30 days of the transaction date.",
    "refund_time"  : "Approved refunds are credited within 5-7 business days.",
    "fraud_report" : "To report fraud, call 1800-XXX-XXXX (24/7) or email fraud@fintech.com.",
    "kyc"          : "KYC update needs Aadhaar + PAN. Go to app > Profile > KYC Update.",
}

REFUND_POLICY = {
    "eligible_window_days" : 30,
    "processing_days"      : 7,
    "conditions"           : ["Item not received", "Duplicate charge",
                              "Merchant error", "Unauthorised transaction"],
}

print(f"Loaded {len(TRANSACTION_DB)} transactions | {len(FAQ_KB)} FAQs")


def classify_query(query: str) -> str:
    q = query.lower()
    if any(k in q for k in ["fraud", "suspicious", "unauthorised", "scam"]):
        return "fraud"
    if any(k in q for k in ["refund", "money back", "return"]):
        return "refund"
    if any(k in q for k in ["transaction", "payment", "txn", "charge"]):
        return "transaction"
    return "faq"

# Test
for q in ["suspicious TXN002", "refund TXN003", "status TXN001", "UPI limit"]:
    print(f"  '{q}' -> {classify_query(q).upper()}")


def retrieval_agent(state: AgentState) -> AgentState:
    print(f"  [Retrieval] Category: {state['category'].upper()}")
    query, category = state["query"], state["category"]
    parts = []

    # Extract TXN ID from query if present
    m      = re.search(r'\bTXN\d{3}\b', query.upper())
    txn_id = m.group(0) if m else None

    if txn_id and txn_id in TRANSACTION_DB:
        parts.append(f"Transaction Record:\n{json.dumps(TRANSACTION_DB[txn_id], indent=2)}")

    if category == "fraud":
        if txn_id and txn_id in TRANSACTION_DB:
            parts.append(f"Fraud Flag: {TRANSACTION_DB[txn_id].get('flag_reason', 'None')}")
        parts.append(f"Fraud Policy: {FAQ_KB['fraud_report']}")

    elif category == "refund":
        parts.append(f"Refund Policy:\n{json.dumps(REFUND_POLICY, indent=2)}")
        if not txn_id:
            pending = [t for t in TRANSACTION_DB.values() if t.get("status") == "refund_requested"]
            if pending:
                parts.append(f"Pending Refunds:\n{json.dumps(pending, indent=2)}")

    elif category == "transaction" and not txn_id:
        parts.append(f"Recent Transactions:\n{json.dumps(list(TRANSACTION_DB.values()), indent=2)}")

    elif category == "faq":
        q = query.lower()
        for key, val in FAQ_KB.items():
            if any(kw in q for kw in key.split("_")):
                parts.append(f"FAQ: {val}")
        if not parts:
            parts.append("\n".join(f"- {v}" for v in FAQ_KB.values()))

    context = "\n\n".join(parts) if parts else "No relevant data found."
    print(f"  Context: {len(context)} chars")
    return {**state, "context": context}

print("Retrieval Agent defined.")


REASONING_SYSTEM = (
    "You are a senior fintech customer support specialist.\n"
    "Analyse the query step-by-step using the context below.\n\n"
    "FORMAT (follow exactly):\n"
    "1. UNDERSTAND  : Restate what the customer is asking.\n"
    "2. ANALYSE     : Examine the context data carefully.\n"
    "3. IDENTIFY    : Note any issues, flags, or policy conditions.\n"
    "4. CONCLUDE    : State the answer clearly.\n"
    "5. RECOMMEND   : Suggest next steps.\n\n"
    "Rules: Use ONLY the provided context. Never guess. "
    "For fraud, always recommend immediate action."
)

def reasoning_agent(state: AgentState) -> AgentState:
    attempt = state["retry_count"] + 1
    print(f"  [Reasoning] Attempt {attempt}/{MAX_RETRIES + 1}")

    user_msg = f"Customer Query: {state['query']}\n\nContext:\n{state['context']}"

    # On retry: attach validation feedback so agent can correct itself
    if state.get("validation_feedback") and state["retry_count"] > 0:
        user_msg += (
            f"\n\nPREVIOUS ANSWER REJECTED.\n"
            f"Feedback: {state['validation_feedback']}\n"
            "Please fix the answer based on this feedback."
        )

    response = call_llm(REASONING_SYSTEM, user_msg)

    # Split reasoning (steps 1-4) from answer (step 5)
    lines, reasoning, answer, in_ans = response.split("\n"), [], [], False
    for line in lines:
        if line.strip().upper().startswith("5.") or "RECOMMEND" in line.upper():
            in_ans = True
        (answer if in_ans else reasoning).append(line)

    return {
        **state,
        "intermediate_reasoning" : "\n".join(reasoning),
        "draft_answer"           : "\n".join(answer) if answer else response,
    }

print("Reasoning Agent defined.")


VALIDATION_SYSTEM = (
    "You are a QA specialist for a fintech support system.\n"
    "Evaluate the answer on: Accuracy, Completeness, Safety, Clarity, Grounding.\n\n"
    "Respond ONLY with this JSON (no extra text, no markdown):\n"
    '{\n'
    '  "passed": true or false,\n'
    '  "score": 0-10,\n'
    '  "issues": ["issue1"],\n'
    '  "feedback": "improvement instructions or empty string"\n'
    '}'
)

def validation_agent(state: AgentState) -> AgentState:
    print(f"  [Validation] Checking answer...")

    user_msg = (
        f"Customer Query: {state['query']}\n\n"
        f"Context:\n{state['context']}\n\n"
        f"Draft Answer:\n{state['draft_answer']}\n\n"
        "Evaluate strictly. Respond ONLY with the JSON."
    )

    raw = call_llm(VALIDATION_SYSTEM, user_msg)

    try:
        raw    = re.sub(r"```json|```", "", raw).strip()
        result = json.loads(raw)
    except (json.JSONDecodeError, ValueError):
        result = {"passed": True, "score": 7, "issues": [], "feedback": ""}

    passed   = result.get("passed", True)
    score    = result.get("score", 7)
    feedback = result.get("feedback", "")

    print(f"  Result: {'PASSED' if passed else 'FAILED'} (score: {score}/10)")

    new_state = {**state, "validation_flag": passed, "validation_feedback": feedback}

    if passed:
        new_state["final_answer"] = state["draft_answer"]
        print("  Answer approved.")
    else:
        new_state["retry_count"] = state["retry_count"] + 1
        print(f"  Retry {new_state['retry_count']}/{MAX_RETRIES}")

    return new_state

print("Validation Agent defined.")


def should_retry_or_end(state: AgentState) -> Literal["reasoning_agent", "end"]:
    if state["validation_flag"]:
        print("  [Router] Passed -> END")
        return "end"
    if state["retry_count"] >= MAX_RETRIES:
        print(f"  [Router] Max retries reached -> END (fallback)")
        return "end"
    print(f"  [Router] Failed -> RETRY {state['retry_count']}/{MAX_RETRIES}")
    return "reasoning_agent"

def apply_fallback(state: AgentState) -> AgentState:
    if not state.get("final_answer"):
        state = {**state, "final_answer": (
            "We could not process your request. Please contact support:\n"
            "  Phone: 1800-XXX-XXXX (24/7)\n"
            "  Email: support@fintech.com"
        )}
    return state

print("Conditional edge and fallback defined.")


graph = StateGraph(AgentState)

# Add nodes
graph.add_node("retrieval_agent",  retrieval_agent)
graph.add_node("reasoning_agent",  reasoning_agent)
graph.add_node("validation_agent", validation_agent)

# Fixed edges
graph.add_edge(START, "retrieval_agent")
graph.add_edge("retrieval_agent", "reasoning_agent")
graph.add_edge("reasoning_agent", "validation_agent")

# Conditional edge (retry loop)
graph.add_conditional_edges(
    "validation_agent",
    should_retry_or_end,
    {"reasoning_agent": "reasoning_agent", "end": END},
)

app = graph.compile()
print("Graph compiled. Nodes: retrieval -> reasoning -> validation -> (retry or END)")


def run_query(query: str) -> dict:
    print(f"\n{'='*55}")
    print(f"Q: {query}")
    print(f"{'='*55}")

    state = app.invoke({
        "query"                  : query,
        "category"               : classify_query(query),
        "context"                : "",
        "intermediate_reasoning" : "",
        "draft_answer"           : "",
        "validation_flag"        : False,
        "validation_feedback"    : "",
        "retry_count"            : 0,
        "final_answer"           : "",
    })

    state = apply_fallback(state)
    print(f"\nA: {state['final_answer']}")
    print(f"   Retries: {state['retry_count']} | Validation: {'PASSED' if state['validation_flag'] else 'FALLBACK'}")
    return state

print("Runner ready.")


queries = [
    "What is the status of my transaction TXN001?",
    "There is a suspicious transaction TXN002 on my account. What should I do?",
    "I want a refund for transaction TXN003. Am I eligible?",
    "What is the UPI transaction limit per day?",
]

for q in queries:
    state = run_query(q)
    print()


Testing Groq...
Hello, it's nice to meet you and I'm here to assist you with any questions or tasks you may have.
AgentState defined.
Loaded 3 transactions | 5 FAQs
  'suspicious TXN002' -> FRAUD
  'refund TXN003' -> REFUND
  'status TXN001' -> TRANSACTION
  'UPI limit' -> FAQ
Retrieval Agent defined.
Reasoning Agent defined.
Validation Agent defined.
Conditional edge and fallback defined.
Graph compiled. Nodes: retrieval -> reasoning -> validation -> (retry or END)
Runner ready.

Q: What is the status of my transaction TXN001?
  [Retrieval] Category: TRANSACTION


NameError: name 're' is not defined